In [1]:
import h5py
import cv2
import numpy as np
import os

path = '/data/fmalmb/DATA/tmp/Testdata.h5'
camera_id = 'cam00'  # Change to cam01, cam02, etc., as needed
output_video = f'{camera_id}_recording.mp4'

with h5py.File(path, 'r') as f:
    frames_group = f[f'rec/{camera_id}/frames']
    frame_keys = sorted(list(frames_group.keys()))
    total_frames = len(frame_keys)
    print(f"Processing {total_frames} frames from {camera_id}...")

    first_bytes = frames_group[f'{frame_keys[0]}/color'][()]
    first_buffer = np.frombuffer(first_bytes, dtype=np.uint8)
    first_img = cv2.imdecode(first_buffer, cv2.IMREAD_COLOR)
    if first_img is None:
        raise ValueError("Could not decode the first frame. Check your data.")
    height, width, layers = first_img.shape
    print(f"Detected resolution: {width}x{height}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_video, fourcc, 30.0, (width, height))
    try:
        for index, frame_key in enumerate(frame_keys):
            compressed_bytes = frames_group[f'{frame_key}/color'][()]
            img_buffer = np.frombuffer(compressed_bytes, dtype=np.uint8)
            img = cv2.imdecode(img_buffer, cv2.IMREAD_COLOR)
            if img is not None:
                video_writer.write(img)
            else:
                print(f"Warning: Failed to decode frame {frame_key}. Skipping.")
            if (index + 1) % 20 == 0 or (index + 1) == total_frames:
                print(f"  Progress: {index + 1}/{total_frames} frames processed.")
    finally:
        video_writer.release()
        print(f"\nSuccess! Saved video file as: {os.path.abspath(output_video)}")


Processing 128 frames from cam00...
Detected resolution: 3360x1890
  Progress: 20/128 frames processed.
  Progress: 40/128 frames processed.
  Progress: 60/128 frames processed.
  Progress: 80/128 frames processed.
  Progress: 100/128 frames processed.
  Progress: 120/128 frames processed.
  Progress: 128/128 frames processed.

Success! Saved video file as: /home/fmalmb/CODE/oak_recorder/cam00_recording.mp4


In [9]:
import h5py
import cv2
import numpy as np
import os

path = '/data/fmalmb/DATA/tmp/Testdata.h5'
output_video = 'multi_camera_4k_grid.mp4'
CANVAS_WIDTH = 3840
CANVAS_HEIGHT = 2160
SLOT_WIDTH = 1280
SLOT_HEIGHT = 720

with h5py.File(path, 'r') as f:
    rec_group = f['rec']
    camera_ids = sorted(list(rec_group.keys()))
    frame_keys = sorted(list(rec_group[camera_ids[0]]['frames'].keys()))
    total_frames = len(frame_keys)
    print(f"Detected {len(camera_ids)} cameras with {total_frames} frames each.")
    print(f"Creating a 3x3 grid video at 4K resolution ({CANVAS_WIDTH}x{CANVAS_HEIGHT})...")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_video, fourcc, 30.0, (CANVAS_WIDTH, CANVAS_HEIGHT))
    try:
        for idx, frame_key in enumerate(frame_keys):
            canvas = np.zeros((CANVAS_HEIGHT, CANVAS_WIDTH, 3), dtype=np.uint8)
            for cam_idx, cam_id in enumerate(camera_ids):
                try:
                    compressed_bytes = rec_group[f'{cam_id}/frames/{frame_key}/color'][()]
                    img_buffer = np.frombuffer(compressed_bytes, dtype=np.uint8)
                    img = cv2.imdecode(img_buffer, cv2.IMREAD_COLOR)
                    if img is not None:
                        resized_img = cv2.resize(img, (SLOT_WIDTH, SLOT_HEIGHT), interpolation=cv2.INTER_AREA)
                        row, col = cam_idx // 3, cam_idx % 3
                        y1, x1 = row * SLOT_HEIGHT, col * SLOT_WIDTH
                        canvas[y1:y1 + SLOT_HEIGHT, x1:x1 + SLOT_WIDTH] = resized_img
                except Exception as e:
                    print(f"Warning: Failed to process frame {frame_key} on {cam_id}: {e}")
            video_writer.write(canvas)
            if (idx + 1) % 20 == 0 or (idx + 1) == total_frames:
                print(f"  Progress: {idx + 1}/{total_frames} grid frames compiled.")
    finally:
        video_writer.release()
        print(f"\nSuccess! 4K grid video saved to: {os.path.abspath(output_video)}")


Detected 7 cameras with 128 frames each.
Creating a 3x3 grid video at 4K resolution (3840x2160)...
  Progress: 20/128 grid frames compiled.
  Progress: 40/128 grid frames compiled.
  Progress: 60/128 grid frames compiled.
  Progress: 80/128 grid frames compiled.
  Progress: 100/128 grid frames compiled.
  Progress: 120/128 grid frames compiled.
  Progress: 128/128 grid frames compiled.

Success! 4K grid video saved to: /home/fmalmb/CODE/oak_recorder/multi_camera_4k_grid.mp4


In [10]:
import h5py
import cv2
import numpy as np
import json
import mediapipe as mp

path = '/data/fmalmb/DATA/tmp/Testdata.h5'
output_json = 'extracted_2d_keypoints.json'

# Downscale width for hand model (full frame is 3360x1890).
HAND_DETECT_MAX_WIDTH = 1280

mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
    (5, 9), (9, 13), (13, 17),
]


def extract_pose_landmarks(pose_landmarks, w, h):
    out = []
    for lm_id, lm in enumerate(pose_landmarks.landmark):
        out.append({
            'id': lm_id,
            'x': float(lm.x * w),
            'y': float(lm.y * h),
            'confidence': float(lm.visibility),
        })
    return out


# MediaPipe pose wrist indices — label hands by proximity (MP handedness is often swapped).
POSE_LEFT_WRIST = 15
POSE_RIGHT_WRIST = 16


def assign_hands_by_pose_wrists(detected_hands, body_landmarks):
    """Pick left/right hand by 2D distance to pose wrists, not MediaPipe handedness."""
    if not detected_hands:
        return [], []
    if len(body_landmarks) > POSE_RIGHT_WRIST:
        lw = body_landmarks[POSE_LEFT_WRIST]
        rw = body_landmarks[POSE_RIGHT_WRIST]
        if lw['confidence'] >= 0.3 and rw['confidence'] >= 0.3:
            if len(detected_hands) == 1:
                hx, hy = detected_hands[0][0]['x'], detected_hands[0][0]['y']
                dl = (hx - lw['x']) ** 2 + (hy - lw['y']) ** 2
                dr = (hx - rw['x']) ** 2 + (hy - rw['y']) ** 2
                return (detected_hands[0], []) if dl <= dr else ([], detected_hands[0])
            h0, h1 = detected_hands[0], detected_hands[1]
            x0, y0 = h0[0]['x'], h0[0]['y']
            x1, y1 = h1[0]['x'], h1[0]['y']
            d0l = (x0 - lw['x']) ** 2 + (y0 - lw['y']) ** 2
            d0r = (x0 - rw['x']) ** 2 + (y0 - rw['y']) ** 2
            d1l = (x1 - lw['x']) ** 2 + (y1 - lw['y']) ** 2
            d1r = (x1 - rw['x']) ** 2 + (y1 - rw['y']) ** 2
            if d0l + d1r <= d0r + d1l:
                return h0, h1
            return h1, h0
    if len(detected_hands) == 1:
        return detected_hands[0], []
    return detected_hands[0], detected_hands[1]


def run_hands_full_frame(img_bgr):
    """Detect hands on every camera view; return list of 21-landmark hands."""
    h, w = img_bgr.shape[:2]
    scale = min(1.0, HAND_DETECT_MAX_WIDTH / float(w))
    if scale < 1.0:
        small = cv2.resize(img_bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    else:
        small = img_bgr
    sh, sw = small.shape[:2]
    result = hands.process(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
    if not result.multi_hand_landmarks:
        return []
    inv = 1.0 / scale
    detected = []
    for hand_lms, handedness in zip(result.multi_hand_landmarks, result.multi_handedness):
        score = float(handedness.classification[0].score)
        detected.append([
            {
                'id': i,
                'x': float(lm.x * sw * inv),
                'y': float(lm.y * sh * inv),
                'confidence': float(max(score, 1.0 - abs(lm.z))),
            }
            for i, lm in enumerate(hand_lms.landmark)
        ])
    return detected


tracking_results = {}

with h5py.File(path, 'r') as f:
    rec_group = f['rec']
    camera_ids = sorted(list(rec_group.keys()))
    frame_keys = sorted(list(rec_group[camera_ids[0]]['frames'].keys()))
    print(f'Starting pose + hand extraction on {len(camera_ids)} cameras...')
    for idx, frame_key in enumerate(frame_keys):
        tracking_results[frame_key] = {}
        for cam_id in camera_ids:
            try:
                compressed_bytes = rec_group[f'{cam_id}/frames/{frame_key}/color'][()]
                img_buffer = np.frombuffer(compressed_bytes, dtype=np.uint8)
                img = cv2.imdecode(img_buffer, cv2.IMREAD_COLOR)
                if img is None:
                    continue
                h, w, _ = img.shape
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                pose_result = pose.process(img_rgb)
                body = []
                if pose_result.pose_landmarks:
                    body = extract_pose_landmarks(pose_result.pose_landmarks, w, h)
                left_hand, right_hand = [], []
                detected_hands = run_hands_full_frame(img)
                if detected_hands and body:
                    left_hand, right_hand = assign_hands_by_pose_wrists(detected_hands, body)
                tracking_results[frame_key][cam_id] = {
                    'body': body,
                    'left_hand': left_hand,
                    'right_hand': right_hand,
                }
            except Exception as e:
                print(f'Error processing {cam_id} @ frame {frame_key}: {e}')
                tracking_results[frame_key][cam_id] = {
                    'body': [], 'left_hand': [], 'right_hand': [],
                }
        if (idx + 1) % 20 == 0 or (idx + 1) == len(frame_keys):
            print(f'  Progress: {idx + 1}/{len(frame_keys)} frames inferred.')

pose.close()
hands.close()

with open(output_json, 'w') as out_f:
    json.dump(tracking_results, out_f, indent=2)
print(f'Saved pose + hands to {output_json}')


Starting pose + hand extraction on 7 cameras...


I0000 00:00:1780518213.912780 2308008 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1780518213.947150 2313421 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.159.04), renderer: NVIDIA GeForce RTX 4090/PCIe/SSE2
I0000 00:00:1780518213.949650 2308008 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1780518213.975585 2313454 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.159.04), renderer: NVIDIA GeForce RTX 4090/PCIe/SSE2
W0000 00:00:1780518213.984073 2313436 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780518213.987729 2313390 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780518213.999416 2313445 inference_feedback_manager.cc:114] Feedback manager requires a model with a 

  Progress: 20/128 frames inferred.
  Progress: 40/128 frames inferred.
  Progress: 60/128 frames inferred.
  Progress: 80/128 frames inferred.
  Progress: 100/128 frames inferred.
  Progress: 120/128 frames inferred.
  Progress: 128/128 frames inferred.
Saved pose + hands to extracted_2d_keypoints.json


In [1]:
# Load Chameleon calibration from H5 and build P = K @ [R|t] per camera.
# Run this cell before triangulation.
import h5py
import pickle
import types
import sys
import numpy as np
import cv2

H5_PATH = '/data/fmalmb/DATA/tmp/Testdata.h5'


class ChameleonCalibrationV1:
    """Stub for unpickling when calib package is not installed."""
    pass


calib_mod = types.ModuleType('calib.chameleon_calibration')
calib_mod.ChameleonCalibrationV1 = ChameleonCalibrationV1
sys.modules['calib.chameleon_calibration'] = calib_mod
sys.modules['calib'] = types.ModuleType('calib')


def build_camera_cal(intrin_vec, cam_to_world):
    cx, cy, fx, fy = intrin_vec[2], intrin_vec[3], intrin_vec[4], intrin_vec[5]
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
    dist = intrin_vec[6:14].astype(np.float64)
    T_wc = np.linalg.inv(cam_to_world)
    P = K @ np.hstack([T_wc[:3, :3], T_wc[:3, 3:4]])
    return K, dist, P


def undistort_pixel(x, y, K, dist):
    pt = np.array([[[x, y]]], dtype=np.float64)
    out = cv2.undistortPoints(pt, K, dist, P=K)
    return float(out[0, 0, 0]), float(out[0, 0, 1])


with h5py.File(H5_PATH, 'r') as f:
    cal = pickle.loads(bytes(f['calibration/cal_pickle'][()]))
    camera_ids = sorted(f['rec'].keys())

PROJECTION_MATRICES = {}
CAMERA_CALIB = {}

for i, cam_id in enumerate(camera_ids):
    if i >= cal.n_cameras:
        raise ValueError(f'No calibration entry for {cam_id} (index {i})')
    K, dist, P = build_camera_cal(cal.intrinsics[i], cal.cam_to_world[i])
    PROJECTION_MATRICES[cam_id] = P
    w, h = cal.intrinsics[i][0], cal.intrinsics[i][1]
    CAMERA_CALIB[cam_id] = {'K': K, 'dist': dist, 'width': float(w), 'height': float(h)}
    print(
        f"  {cam_id}: serial={cal.serial[i]}, resolution={int(w)}x{int(h)}, "
        f"fx={K[0, 0]:.1f}, cx={K[0, 2]:.1f}, cy={K[1, 2]:.1f}"
    )

CAMERA_WORLD_TRANSFORMS = {}
for i, cam_id in enumerate(camera_ids):
    CAMERA_WORLD_TRANSFORMS[cam_id] = cal.cam_to_world[i].astype(np.float64)

print(f'Loaded ChameleonCalibrationV1 ({cal.n_cameras} cameras, gravity={cal.gravity_world})')


  cam00: serial=1844301021BD6C0E00, resolution=3360x1890, fx=2014.5, cx=1673.0, cy=918.5
  cam01: serial=18443010619BDB0E00, resolution=3360x1890, fx=1997.1, cx=1690.5, cy=956.0
  cam02: serial=18443010C1146C0E00, resolution=3360x1890, fx=1990.6, cx=1723.1, cy=937.1
  cam03: serial=19443010C1681A1300, resolution=3360x1890, fx=2011.3, cx=1700.2, cy=926.9
  cam04: serial=18443010D1AE630E00, resolution=3360x1890, fx=1995.0, cx=1697.7, cy=953.3
  cam05: serial=18443010C1FADA0E00, resolution=3360x1890, fx=2014.8, cx=1728.1, cy=950.2
  cam06: serial=184430100145640E00, resolution=3360x1890, fx=2002.5, cx=1679.1, cy=964.0
Loaded ChameleonCalibrationV1 (7 cameras, gravity=[0. 1. 0.])


In [5]:
# Multi-view DLT triangulation (body + hands).
import json
import numpy as np

CONFIDENCE_THRESHOLD = 0.60
HAND_CONFIDENCE_THRESHOLD = 0.90
UNDISTORT_PIXELS = True
OUTPUT_JSON = 'final_reconstructed_3d_keypoints.json'

# Global joint IDs in output JSON (hands need 2+ views like body)
PARTS = [
    ('body', 33, 0, CONFIDENCE_THRESHOLD, 2),
    ('left_hand', 21, 100, HAND_CONFIDENCE_THRESHOLD, 2),
    ('right_hand', 21, 200, HAND_CONFIDENCE_THRESHOLD, 2),
]


def triangulate_point_dlt(projection_matrices, pixel_points):
    A = []
    for P, (x, y) in zip(projection_matrices, pixel_points):
        A.append(x * P[2, :] - P[0, :])
        A.append(y * P[2, :] - P[1, :])
    A = np.array(A)
    _, _, Vh = np.linalg.svd(A)
    X_h = Vh[-1]
    return X_h[:3] / X_h[3]


def normalize_cam_landmarks(cam_entry):
    """Support legacy list-only body format and new dict format."""
    if isinstance(cam_entry, list):
        return {'body': cam_entry, 'left_hand': [], 'right_hand': []}
    return {
        'body': cam_entry.get('body', []),
        'left_hand': cam_entry.get('left_hand', []),
        'right_hand': cam_entry.get('right_hand', []),
    }


def get_joint(landmarks, joint_idx):
    for lm in landmarks:
        if lm['id'] == joint_idx:
            return lm
    if joint_idx < len(landmarks):
        return landmarks[joint_idx]
    return None


with open('extracted_2d_keypoints.json', 'r') as in_f:
    tracking_data = json.load(in_f)

reconstructed_3d_data = {}
for frame_key, cameras in tracking_data.items():
    reconstructed_3d_data[frame_key] = {}
    for part_name, n_joints, id_offset, conf_thr, min_cams in PARTS:
        for joint_idx in range(n_joints):
            valid_ps, valid_pixels, weights = [], [], []
            for cam_id, cam_entry in cameras.items():
                parts = normalize_cam_landmarks(cam_entry)
                landmarks = parts[part_name]
                if cam_id not in PROJECTION_MATRICES or len(landmarks) == 0:
                    continue
                joint_data = get_joint(landmarks, joint_idx)
                if joint_data is None or joint_data['confidence'] < conf_thr:
                    continue
                x, y = joint_data['x'], joint_data['y']
                if UNDISTORT_PIXELS:
                    K = CAMERA_CALIB[cam_id]['K']
                    dist = CAMERA_CALIB[cam_id]['dist']
                    x, y = undistort_pixel(x, y, K, dist)
                valid_ps.append(PROJECTION_MATRICES[cam_id])
                valid_pixels.append((x, y))
                weights.append(joint_data['confidence'])
            if len(valid_ps) >= min_cams:
                try:
                    point_3d = triangulate_point_dlt(valid_ps, valid_pixels)
                    out_id = id_offset + joint_idx
                    reconstructed_3d_data[frame_key][str(out_id)] = {
                        'X': float(point_3d[0]),
                        'Y': float(point_3d[1]),
                        'Z': float(point_3d[2]),
                        'combined_confidence': float(np.mean(weights)),
                        'n_views': len(valid_ps),
                        'part': part_name,
                    }
                except np.linalg.LinAlgError:
                    pass

with open(OUTPUT_JSON, 'w') as out_3d:
    json.dump(reconstructed_3d_data, out_3d, indent=2)

first_key = sorted(reconstructed_3d_data.keys(), key=lambda x: int(x))[0]
frame0 = reconstructed_3d_data[first_key]
n_body = sum(1 for k in frame0 if int(k) < 100)
n_lh = sum(1 for k in frame0 if 100 <= int(k) < 200)
n_rh = sum(1 for k in frame0 if int(k) >= 200)
print(f'Saved {OUTPUT_JSON} ({len(reconstructed_3d_data)} frames)')
print(f'Frame {first_key}: {n_body} body, {n_lh} left-hand, {n_rh} right-hand joints')


Saved final_reconstructed_3d_keypoints.json (128 frames)
Frame 000000: 27 body, 21 left-hand, 21 right-hand joints


In [6]:
import json
import time
import numpy as np
import viser
from viser import transforms as tf

# Calibration world is Y-up; Viser uses Z-up. Flip world Y so the figure is upright.
R_WORLD_TO_VISER = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0],
    [0.0, -1.0, 0.0],
], dtype=np.float64)
FRUSTUM_SCALE_BASE = 0.10
FRUSTUM_SCALE_PER_M = 0.03

with open('final_reconstructed_3d_keypoints.json', 'r') as f:
    threed_data = json.load(f)

frame_keys = sorted(threed_data.keys(), key=lambda x: int(x))

POSE_CONNECTIONS = [
    (11, 12), (11, 23), (12, 24), (23, 24),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32),
    (0, 1), (1, 2), (2, 3), (0, 4), (4, 5), (5, 6), (3, 7), (6, 8),
    (9, 10),
]
POSE_HAND_JOINTS = {17, 18, 19, 20, 21, 22}
BODY_LEFT_WRIST = 15
BODY_RIGHT_WRIST = 16
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
    (5, 9), (9, 13), (13, 17),
]
LEFT_HAND_OFFSET = 100
RIGHT_HAND_OFFSET = 200
LEFT_HAND_WRIST = LEFT_HAND_OFFSET + 0
RIGHT_HAND_WRIST = RIGHT_HAND_OFFSET + 0


def skeleton_origin(joints_dict):
    """Pelvis midpoint, else mean of all body joints."""
    hip_pts = []
    for hip_id in (23, 24):
        key = str(hip_id)
        if key in joints_dict:
            hip_pts.append([joints_dict[key]['X'], joints_dict[key]['Y'], joints_dict[key]['Z']])
    if len(hip_pts) == 2:
        return np.mean(hip_pts, axis=0)
    body_pts = [
        [v['X'], v['Y'], v['Z']]
        for k, v in joints_dict.items()
        if int(k) < 100
    ]
    if body_pts:
        return np.mean(body_pts, axis=0)
    return np.zeros(3, dtype=np.float64)


def world_to_viser_point(p_world, origin_world):
    return R_WORLD_TO_VISER @ (np.asarray(p_world, dtype=np.float64) - origin_world)


def world_to_viser_pose(T_cw, origin_world):
    T = np.asarray(T_cw, dtype=np.float64).copy()
    T[:3, 3] = R_WORLD_TO_VISER @ (T[:3, 3] - origin_world)
    T[:3, :3] = R_WORLD_TO_VISER @ T[:3, :3]
    wxyz = tf.SO3.from_matrix(T[:3, :3]).wxyz
    return wxyz.astype(np.float64), T[:3, 3].astype(np.float64)


def attach_hands_to_wrists(joint_coords):
    """Snap each hand cluster to the nearest body wrist (fixes swapped MP labels)."""
    mapping = {}
    body_wrists = [w for w in (BODY_LEFT_WRIST, BODY_RIGHT_WRIST) if w in joint_coords]
    hand_specs = [
        (LEFT_HAND_OFFSET, LEFT_HAND_WRIST),
        (RIGHT_HAND_OFFSET, RIGHT_HAND_WRIST),
    ]
    hands_present = [(off, hw) for off, hw in hand_specs if hw in joint_coords]
    if not hands_present or not body_wrists:
        return joint_coords, mapping

    if len(hands_present) == 1:
        off, hw = hands_present[0]
        bw = min(body_wrists, key=lambda w: np.linalg.norm(joint_coords[hw] - joint_coords[w]))
        mapping[off] = bw
        delta = joint_coords[bw] - joint_coords[hw]
        for jid in joint_coords:
            if off <= jid < off + 21:
                joint_coords[jid] = joint_coords[jid] + delta
        return joint_coords, mapping

    if len(body_wrists) == 2:
        (off_a, hw_a), (off_b, hw_b) = hands_present
        w_l, w_r = body_wrists[0], body_wrists[1]
        d_a_l = np.linalg.norm(joint_coords[hw_a] - joint_coords[w_l])
        d_a_r = np.linalg.norm(joint_coords[hw_a] - joint_coords[w_r])
        d_b_l = np.linalg.norm(joint_coords[hw_b] - joint_coords[w_l])
        d_b_r = np.linalg.norm(joint_coords[hw_b] - joint_coords[w_r])
        if d_a_l + d_b_r <= d_a_r + d_b_l:
            assignments = [(off_a, hw_a, w_l), (off_b, hw_b, w_r)]
        else:
            assignments = [(off_a, hw_a, w_r), (off_b, hw_b, w_l)]
        for off, hw, bw in assignments:
            mapping[off] = bw
            delta = joint_coords[bw] - joint_coords[hw]
            for jid in joint_coords:
                if off <= jid < off + 21:
                    joint_coords[jid] = joint_coords[jid] + delta
    return joint_coords, mapping


def joint_display_ids(joint_coords):
    ids = []
    for jid in joint_coords:
        if jid in POSE_HAND_JOINTS:
            continue
        if jid in (LEFT_HAND_WRIST, RIGHT_HAND_WRIST):
            continue
        ids.append(jid)
    return sorted(ids)


def apply_fov(server, fov_deg):
    fov_rad = float(np.deg2rad(fov_deg))
    server.initial_camera.fov = fov_rad
    for client in server.get_clients().values():
        client.camera.fov = fov_rad


def build_joint_coords(joints_dict, origin_world):
    out = {}
    for j_id, coord in joints_dict.items():
        p = world_to_viser_point([coord['X'], coord['Y'], coord['Z']], origin_world)
        out[int(j_id)] = p.astype(np.float32)
    return attach_hands_to_wrists(out)


def hand_bone_connections(hand_offset, body_wrist_id):
    conns = []
    for a, b in HAND_CONNECTIONS:
        ja = body_wrist_id if a == 0 else a + hand_offset
        jb = body_wrist_id if b == 0 else b + hand_offset
        conns.append((ja, jb))
    return conns


def add_bones(server, name, joint_coords, connections, color):
    segs = []
    for a, b in connections:
        if a in joint_coords and b in joint_coords:
            segs.append([joint_coords[a], joint_coords[b]])
    if segs:
        server.scene.add_line_segments(
            name=name,
            points=np.array(segs, dtype=np.float32),
            colors=np.array(color, dtype=np.uint8),
            line_width=2.0,
        )


# Static origin from first frame (keeps rig + subject near 0,0,0 in Viser).
ORIGIN_WORLD = skeleton_origin(threed_data[frame_keys[0]])


def setup_cameras(server):
    if 'CAMERA_WORLD_TRANSFORMS' not in globals():
        print('Warning: run calibration cell first — no camera frustums.')
        return
    for cam_id, T_cw in CAMERA_WORLD_TRANSFORMS.items():
        if cam_id not in CAMERA_CALIB:
            continue
        calib = CAMERA_CALIB[cam_id]
        K = calib['K']
        fov = 2.0 * np.arctan(calib['height'] / (2.0 * K[1, 1]))
        aspect = float(calib['width'] / calib['height'])
        wxyz, position = world_to_viser_pose(T_cw, ORIGIN_WORLD)
        dist = float(np.linalg.norm(position))
        frustum_scale = float(np.clip(FRUSTUM_SCALE_BASE + FRUSTUM_SCALE_PER_M * dist, 0.08, 0.16))
        server.scene.add_camera_frustum(
            name=f'/cameras/{cam_id}',
            fov=float(fov),
            aspect=aspect,
            scale=frustum_scale,
            line_width=0.5,
            color=(80, 160, 255),
            wxyz=wxyz,
            position=position,
        )
        server.scene.add_frame(
            name=f'/cameras/{cam_id}/axis',
            axes_length=0.08,
            axes_radius=0.004,
            wxyz=wxyz,
            position=position,
        )


def main():
    server = viser.ViserServer()
    running = True
    try:
        server.scene.set_up_direction('+z')
        server.scene.configure_environment_map(hdri=None, background=False, environment_intensity=0.6)
        server.gui.configure_theme(dark_mode=True)
        server.scene.set_background_image(np.zeros((8, 8, 3), dtype=np.uint8))

        setup_cameras(server)

        server.initial_camera.position = (2.5, 2.0, 1.6)
        server.initial_camera.look_at = (0.0, 0.0, 0.9)
        server.initial_camera.up = (0.0, 0.0, 1.0)

        print(f'\nViser: open the URL shown above')
        print(f'Origin offset (calibration world): {ORIGIN_WORLD}')

        play_button = server.gui.add_button('Play / Pause')
        stop_button = server.gui.add_button('Stop viewer')
        speed_slider = server.gui.add_slider('Playback Speed (FPS)', min=1, max=60, step=1, initial_value=15)
        frame_slider = server.gui.add_slider('Timeline Frame', min=0, max=len(frame_keys) - 1, step=1, initial_value=0)
        fov_slider = server.gui.add_slider('Camera FOV (deg)', min=20.0, max=120.0, step=1.0, initial_value=75.0)
        apply_fov(server, fov_slider.value)
        is_playing = False

        @play_button.on_click
        def _(_) -> None:
            nonlocal is_playing
            is_playing = not is_playing

        @stop_button.on_click
        def _stop(_) -> None:
            nonlocal running
            running = False

        @fov_slider.on_update
        def _fov(_) -> None:
            apply_fov(server, fov_slider.value)

        current_idx = 0
        while running:
            if not is_playing:
                current_idx = frame_slider.value
            f_key = frame_keys[current_idx]
            joint_coords, hand_wrist_map = build_joint_coords(threed_data[f_key], ORIGIN_WORLD)
            display_ids = joint_display_ids(joint_coords)

            if display_ids:
                pts = np.stack([joint_coords[i] for i in display_ids]).astype(np.float32)
                colors = np.zeros((len(display_ids), 3), dtype=np.uint8)
                for i, jid in enumerate(display_ids):
                    if jid >= RIGHT_HAND_OFFSET:
                        bw = hand_wrist_map.get(RIGHT_HAND_OFFSET, BODY_RIGHT_WRIST)
                    elif jid >= LEFT_HAND_OFFSET:
                        bw = hand_wrist_map.get(LEFT_HAND_OFFSET, BODY_LEFT_WRIST)
                    else:
                        bw = None
                    if bw == BODY_LEFT_WRIST:
                        colors[i] = (255, 200, 0)
                    elif bw == BODY_RIGHT_WRIST:
                        colors[i] = (255, 120, 0)
                    else:
                        colors[i] = (220, 220, 220)
                server.scene.add_point_cloud(
                    name='/skeleton/joints',
                    points=pts,
                    colors=colors,
                    point_size=0.005,
                )

            add_bones(server, '/skeleton/body', joint_coords, POSE_CONNECTIONS, (0, 255, 120))
            if any(LEFT_HAND_OFFSET <= jid < LEFT_HAND_OFFSET + 21 for jid in joint_coords):
                bw = hand_wrist_map.get(LEFT_HAND_OFFSET, BODY_LEFT_WRIST)
                color = (255, 180, 0) if bw == BODY_LEFT_WRIST else (255, 120, 0)
                add_bones(server, '/skeleton/left_hand', joint_coords, hand_bone_connections(LEFT_HAND_OFFSET, bw), color)
            if any(RIGHT_HAND_OFFSET <= jid < RIGHT_HAND_OFFSET + 21 for jid in joint_coords):
                bw = hand_wrist_map.get(RIGHT_HAND_OFFSET, BODY_RIGHT_WRIST)
                color = (255, 180, 0) if bw == BODY_LEFT_WRIST else (255, 120, 0)
                add_bones(server, '/skeleton/right_hand', joint_coords, hand_bone_connections(RIGHT_HAND_OFFSET, bw), color)

            if is_playing:
                current_idx = (current_idx + 1) % len(frame_keys)
                frame_slider.value = current_idx
                time.sleep(1.0 / speed_slider.value)
            else:
                time.sleep(0.05)
    except KeyboardInterrupt:
        print('Viewer interrupted.')
    finally:
        server.stop()
        print('Viser stopped.')


main()


╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯


Viser: open the URL shown above
Origin offset (calibration world): [0.00249324 1.57398606 1.68523459]


(viser) Connection opened (0, 1 total), 84 persistent messages

(viser) Connection closed (0, 0 total)

Viewer interrupted.


(viser) Server stopped

Viser stopped.


In [31]:
# Optional: stop a Viser server if the viewer cell did not shut down cleanly.
# Re-run the viewer cell (cell above) after stopping any old server.
print('Use Stop viewer in Viser GUI, or interrupt the viewer cell (Kernel -> Interrupt).')

╭────── viser (listening *:8090) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8090   │
│   Websocket │ ws://localhost:8090     │
│             ╵                         │
╰───────────────────────────────────────╯

Interrupt received. Stopping server...


(viser) Server stopped